# 03 — StarDist with isolated sample-per-GPU workers

Each sample runs in a fresh TensorFlow process with one GPU. Four samples can use four GPUs when measured RAM budgets permit; this is not four-GPU inference of one slide. Review coordinates and fresh QC in the CSV first. Unknown RAM/job limits concurrency to one pilot.

In [ ]:
from pathlib import Path
import os, sys, json
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "vhd" / "control").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter in the extracted pipeline folder (or its notebooks folder).")
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from vhd.control.manifest import load_project, check_policy, save_plan, sample_layout
MANIFEST = Path(os.environ.get("VHD_MANIFEST", ROOT / "config" / "samples.csv"))
SETTINGS = Path(os.environ.get("VHD_SETTINGS", ROOT / "config" / "settings.json"))
if not MANIFEST.exists() or not SETTINGS.exists():
    raise FileNotFoundError("Copy a supplied samples.*.csv to config/samples.csv and settings.g5_24xlarge.example.json to config/settings.json; edit paths and policy first.")
PROJECT = load_project(MANIFEST, SETTINGS)
check_policy(PROJECT)
# Default is a dry run. Set True here only after reviewing the printed plan.
EXECUTE = os.environ.get("VHD_EXECUTE", "0") == "1"
# Optional pilot selection, e.g. ["StudyLegacy__Sample01"]. None selects all applicable rows.
SAMPLE_KEYS = None


## Run only actions applicable to each CSV row

In [ ]:
from vhd.compute.launch import launch_samples
launch_samples(PROJECT, ['stardist'], execute=EXECUTE, sample_keys=SAMPLE_KEYS)